In [1]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START
from langgraph.types import Command
from langgraph.constants import END
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gpt-4o")

c:\Users\21566\anaconda3\envs\ai-langchain1.0\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [21]:
from langchain.messages import AnyMessage,SystemMessage, ToolMessage
from typing_extensions import TypedDict, Annotated
import operator
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add] # 自动追加新消息
    llm_calls: int # 记录 LLM 调用次数

In [22]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
# 初始化模型
model = init_chat_model("gpt-4o")
# 定义工具函数
@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`."""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Divide `a` by `b`."""
    return a / b

# 绑定工具到模型
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [23]:
# 定义节点, 调用模型，模型会根据用户的问题选择不同的工具，不会自动去执行这个工具，function_calling
def llm_call(state:MessagesState):
    """模型调用的节点"""
    response = model_with_tools.invoke(
        [SystemMessage(content="你是一个简易的计算器，可以使用 add, multiply, divide 函数进行计算。")]+
        state["messages"]
    )
    
    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1
    }
    

In [24]:
def tool_node(state:MessagesState):
    """工具调用的节点"""
    last_msg = state["messages"][-1]
    """
    content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 127, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.0004875, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0004875, 'upstream_inference_prompt_cost': 0.0003175, 'upstream_inference_completions_cost': 0.00017}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_0a4b2d353e', 'id': 'gen-1792130430-EWKe7MdEsq3eDIbrSaVY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1434c-3313-71b3-ad0f-e0490b69ed6a-0' tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 2}, 'id': 'call_hb1yV8qPLHPkznyawH2VKiL1', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 127, 'output_tokens': 17, 'total_tokens': 144, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

    """
    messages = []
    # 根据最后一条消息中的工具调用列表，依次调用工具
    for tool_call in last_msg.tool_calls:
        tool_name = tool_call["name"] #获取工具名称
        tool = tools_by_name[tool_name] # 获取工具函数
        args = tool_call["args"] # 获取工具函数参数
        result = tool.invoke(args) # 执行工具
        messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"])) # 将工具结果添加到消息列表中
    
    return {
        "messages": messages,
    }

In [25]:
def should_continue(state:MessagesState)->Literal["tool_node", END]:
    """判断是否继续调用工具"""
    last_msg = state["messages"][-1]
    # 判断模型回复消息列表中最后一条消息中是否需要调用工具
    if last_msg.tool_calls and len(last_msg.tool_calls) > 0:
        return "tool_node"
    else:
        return END

In [26]:
builder = StateGraph(MessagesState)
builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue)
builder.add_edge("tool_node", "llm_call")

agent = builder.compile()

In [27]:
res = agent.invoke({"messages": [{"role": "user", "content": "What is 2 + 2?"}]})
print(res)

{'messages': [{'role': 'user', 'content': 'What is 2 + 2?'}, AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 127, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.0004875, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0004875, 'upstream_inference_prompt_cost': 0.0003175, 'upstream_inference_completions_cost': 0.00017}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_0a4b2d353e', 'id': 'gen-1792130827-QHZTx5LsV04zuvLURw3j', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_ru

In [30]:
for msg in res["messages"]:
    print(msg)

{'role': 'user', 'content': 'What is 2 + 2?'}
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 127, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.0004875, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0004875, 'upstream_inference_prompt_cost': 0.0003175, 'upstream_inference_completions_cost': 0.00017}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_0a4b2d353e', 'id': 'gen-1792130827-QHZTx5LsV04zuvLURw3j', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a14352-4037-74d0-8f1c-0